### Configuração do LDA

In [1]:
import pandas as pd
import gensim
from gensim import corpora
from gensim.models import LdaModel
import nltk
import re

In [2]:
# Carrega o dataset
df_novo = pd.read_csv('meu_arquivo.csv.zip', compression='zip')

In [3]:
documents = df_novo['abstract'].dropna().astype(str).tolist()

In [13]:
# Usar esse caso o dataset esteja pre-processado
def tokenize_text(text):
    """
    Função para tokenizar o texto.
    """

    # Converte para minúsculas e separa em palavras
    tokens = text.lower().split()

    return tokens

# Usar esse caso o dataset esteja pre-processado
def preprocess_text(text):
    """
    Função para limpar e tokenizar o texto.
    """
    # Remove pontuação e caracteres especiais
    text = re.sub(r'[^\w\s]', '', text)

    # Remove as palavras "text", "patente" e "lang"
    text = re.sub(r'\b(text|patente|lang|pt)\b', '', text, flags=re.IGNORECASE)

    # Remove números
    text = re.sub(r'\d+', '', text)
    
    # Converte para minúsculas
    text = text.lower()
    
    # Tokeniza o texto
    tokens = text.split()
    
    # Remove stopwords
    stopwords = set(nltk.corpus.stopwords.words('portuguese'))
    tokens = [token for token in tokens if token not in stopwords]
    
    return tokens

# Pre-processa os documentos
processed_docs = [preprocess_text(doc) for doc in documents]

# Ou, se o dataset já estiver pre-processado, use:
# processed_docs = [tokenize_text(doc) for doc in documents]

# Cria dicionário e corpus
# O dicionário mapeia cada palavra a um ID único.
dictionary = corpora.Dictionary(processed_docs)

# O corpus é a representação Bag-of-Words de cada documento.
# Para cada documento, ele contém uma lista de tuplas (ID_da_palavra, frequência).
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

In [14]:
print(f"Número de documentos: {len(corpus)}")
print(f"Tamanho do dicionário: {len(dictionary)}")
print(f"Exemplo de documento (Bag-of-Words): {corpus[0]}")
print(f"processed_docs[0]: {processed_docs[0]}")

Número de documentos: 25240
Tamanho do dicionário: 52371
Exemplo de documento (Bag-of-Words): [(0, 6), (1, 1), (2, 3), (3, 1), (4, 1), (5, 1), (6, 8), (7, 2), (8, 1), (9, 7), (10, 1), (11, 3), (12, 4), (13, 5), (14, 1), (15, 1), (16, 1), (17, 2), (18, 3), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 5), (31, 2), (32, 1), (33, 4), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 2), (41, 5), (42, 5), (43, 1), (44, 1)]
processed_docs[0]: ['dispositivo', 'sedimentação', 'invenção', 'refere', 'dispositivo', 'sedimentação', 'material', 'contido', 'líquido', 'particular', 'precipitações', 'pluviais', 'elemento', 'sedimentação', 'inserido', 'posição', 'uso', 'elemento', 'poço', 'sendo', 'elemento', 'sedimentação', 'apresenta', 'câmara', 'entrada', 'delimitada', 'parede', 'lateral', 'câmara', 'entrada', 'equipada', 'abertura', 'entrada', 'lateral', 'lado', 'inferior', 'apresenta', 'abertura', 'saída', 'sendo', 'abertura', 'sa

### Treinamento do modelo

In [15]:
# Treina o modelo LDA
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=5,
    random_state=42,
    passes=10,
    alpha='auto', # (Document-Topic Density): Controla a mistura de tópicos por documento
    eta='auto' # (Topic-Word Density): Controla a mistura de palavras por tópico
)

In [16]:
# Exibe os tópicos encontrados
for idx, topic in lda_model.print_topics(-1):
    print(f'Tópico {idx}: {topic}')

Tópico 0: 0.011*"sistema" + 0.009*"presente" + 0.009*"modelo" + 0.008*"portas" + 0.008*"porta" + 0.008*"segurança" + 0.008*"disposição" + 0.007*"conjunto" + 0.006*"utilidade" + 0.006*"dispositivo"
Tópico 1: 0.012*"superior" + 0.011*"sendo" + 0.011*"inferior" + 0.008*"fixação" + 0.007*"base" + 0.007*"parede" + 0.007*"forma" + 0.007*"laterais" + 0.006*"duas" + 0.006*"perfil"
Tópico 2: 0.037*"água" + 0.011*"sistema" + 0.010*"caixa" + 0.010*"válvula" + 0.008*"saída" + 0.008*"descarga" + 0.008*"tubo" + 0.007*"ar" + 0.007*"sanitário" + 0.007*"entrada"
Tópico 3: 0.013*"menos" + 0.013*"dispositivo" + 0.012*"elemento" + 0.012*"posição" + 0.011*"primeira" + 0.010*"porta" + 0.010*"invenção" + 0.009*"segunda" + 0.008*"primeiro" + 0.008*"compreende"
Tópico 4: 0.012*"construção" + 0.010*"material" + 0.010*"invenção" + 0.009*"concreto" + 0.009*"presente" + 0.008*"sistema" + 0.008*"processo" + 0.006*"camada" + 0.005*"estrutura" + 0.005*"revestimento"


In [17]:
import matplotlib.pyplot as plt

# Extrai os tópicos e palavras
topics = lda_model.show_topics(num_topics=-1, num_words=10, formatted=False)

fig, axes = plt.subplots(len(topics), 1, figsize=(10, 2 * len(topics)), constrained_layout=True)

for i, (topic_idx, terms) in enumerate(topics):
    words = [w for w, p in terms]
    probs = [p for w, p in terms]
    axes[i].barh(words, probs)
    axes[i].set_title(f'Tópico {topic_idx}')
    axes[i].invert_yaxis()
    axes[i].set_xlabel('% de importância')
    axes[i].set_xlim(0, max(probs)*1.1)

plt.savefig('lda_topicos.png', dpi=300)
plt.close()

print("Gráfico salvo como 'lda_topicos.png'")

Gráfico salvo como 'lda_topicos.png'
